# Conditional GAN — Drug Molecule Generation from Scratch

**Goal:** Generate novel drug-like molecules (SMILES strings + atom composition) conditioned on 15 molecular properties from the ChEMBL pharmaceutical database.

**Architecture:**
- **Generator:** Custom Autoregressive GRU cell (ARGRUCell) for character-by-character SMILES generation, conditioned on molecular properties
- **Discriminator:** Conv1D branch (SMILES) + Dense branches (atom features, conditions) merged with MiniBatch Discrimination to prevent mode collapse

**Engineering decisions:**
- Mixed precision (float16) for memory efficiency on T4 GPUs
- Multi-GPU MirroredStrategy (2× T4, 32GB total)
- IQR-based outlier clipping, RDKit chemical validity checking
- No pretrained models — all architectures built from scratch

**Kaggle setup:** GPU Accelerator → T4 × 2

## 1. GPU Setup & Mixed Precision

In [1]:
import tensorflow as tf
from tensorflow.keras import mixed_precision

# Enable mixed precision — uses float16 for computation, float32 for variables
mixed_precision.set_global_policy('mixed_float16')

# Configure GPU memory growth to avoid OOM on first allocation
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f'✅ GPUs available: {len(gpus)}')
        for g in gpus:
            print(f'   {g}')
    except RuntimeError as e:
        print(f'⚠️  GPU setup error: {e}')
else:
    print('❌ No GPU found — check Kaggle accelerator settings (Settings → Accelerator → GPU T4 x2)')

2026-06-15 11:12:49.146817: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781521969.324378      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781521969.379522      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781521969.784354      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781521969.784398      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781521969.784401      58 computation_placer.cc:177] computation placer alr

✅ GPUs available: 2
   PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
   PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')


## 2. Imports

In [2]:
import pandas as pd
import numpy as np
import re
import os
from collections import defaultdict, deque
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.layers import Dense, Concatenate, RepeatVector, Dropout, Lambda
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
!pip install rdkit

# Suppress RDKit non-critical warnings
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')
from rdkit import Chem

print('All imports successful.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 51.2 MB/s eta 0:00:00:00:0100:01
All imports successful.


## 3. Data Loading

Dataset: ChEMBL compound database (2-part CSV with semicolon separator).

In [3]:
COLUMN_NAMES = [
    'ChEMBL ID', 'Name', 'Synonyms', 'Type', 'Max Phase',
    'Molecular Weight', 'Targets', 'Bioactivities', 'AlogP',
    'Polar Surface Area', 'HBA', 'HBD', '#RO5 Violations',
    '#Rotatable Bonds', 'Passes Ro3', 'QED Weighted', 'CX Acidic pKa',
    'CX Basic pKa', 'CX LogP', 'CX LogD', 'Aromatic Rings',
    'Structure Type', 'Inorganic Flag', 'Heavy Atoms', 'HBA (Lipinski)',
    'HBD (Lipinski)', '#RO5 Violations (Lipinski)',
    'Molecular Weight (Monoisotopic)', 'Np Likeness Score',
    'Molecular Species', 'Molecular Formula', 'Smiles', 'Inchi Key',
    'Inchi', 'Withdrawn Flag', 'Orphan', 'Records Key', 'Records Name'
]

CSV_KWARGS = dict(sep=';', quotechar='"', low_memory=False, on_bad_lines='skip')

df0 = pd.read_csv('/kaggle/input/datasets/mohamednasra/compounds/DOWNLOAD-heRwUfDRJj-nAMSjmA31y8RyPmnAG6YgOpVpkx4anHU.csv', **CSV_KWARGS)
df1 = pd.read_csv('/kaggle/input/datasets/mohamednasra/compounds/DOWNLOAD-heRwUfDRJj-nAMSjmA31y8RyPmnAG6YgOpVpkx4anHU_part2.csv',
                  header=None, names=COLUMN_NAMES, **CSV_KWARGS)

df_raw = pd.concat([df0, df1], ignore_index=True)
print(f'Combined dataset shape: {df_raw.shape}')

Combined dataset shape: (2494509, 38)


## 4. Feature Selection & SMILES Validation

In [4]:
KEEP_COLS = [
    'Molecular Weight', 'AlogP', 'Polar Surface Area', 'HBA', 'HBD',
    '#RO5 Violations', '#Rotatable Bonds', 'Passes Ro3', 'QED Weighted',
    'CX LogP', 'CX LogD', 'Aromatic Rings', 'Heavy Atoms',
    'Np Likeness Score', 'Molecular Species', 'Molecular Formula', 'Smiles'
]

DF = df_raw[KEEP_COLS].copy().drop_duplicates()

# Drop rows missing essential columns
DF = DF.dropna(subset=['Smiles', 'Molecular Formula', 'Molecular Weight', 'Heavy Atoms'])

# Validate SMILES with RDKit — keep only chemically valid structures
def is_valid_smiles(smi):
    try:
        return Chem.MolFromSmiles(str(smi)) is not None
    except:
        return False

DF = DF[DF['Smiles'].apply(is_valid_smiles)].reset_index(drop=True)
print(f'Valid molecules: {len(DF):,}')

Valid molecules: 2,407,622


## 5. Missing Value Imputation

In [5]:
# Continuous properties: fill with median
FILL_MEDIAN = ['AlogP', 'Polar Surface Area', '#Rotatable Bonds', 'QED Weighted',
               'CX LogP', 'CX LogD', 'Aromatic Rings', 'Np Likeness Score']
for col in FILL_MEDIAN:
    DF[col] = DF[col].fillna(DF[col].median())

# Count properties: fill with 0
FILL_ZERO = ['HBA', 'HBD', '#RO5 Violations']
for col in FILL_ZERO:
    DF[col] = DF[col].fillna(0)

# Molecular Species: infer from molecular formula before falling back to mode
def infer_species(row):
    """Infer ionic species from molecular formula charge indicators."""
    try:
        formula = str(row['Molecular Formula'])
        if '+' in formula and '-' in formula:
            return 'ZWITTERION'
        elif '+' in formula:
            return 'BASE'
        elif '-' in formula:
            return 'ACID'
        else:
            return 'NEUTRAL'
    except:
        return None

mask = DF['Molecular Species'].isna()
DF.loc[mask, 'Molecular Species'] = DF[mask].apply(infer_species, axis=1)

# Final fallback: mode
mode_species = DF['Molecular Species'].mode().iloc[0]
DF['Molecular Species'] = DF['Molecular Species'].fillna(mode_species)
print('Imputation complete.')
print(DF['Molecular Species'].value_counts())

Imputation complete.
Molecular Species
NEUTRAL       1776833
BASE           317308
ACID           270127
ZWITTERION      43354
Name: count, dtype: int64


## 6. Outlier Clipping (IQR)

Replace outliers with nearest boundary value — no data is dropped.

In [6]:
numerical_df = DF.select_dtypes(include=['number'])

for col in numerical_df.columns:
    Q1  = DF[col].quantile(0.25)
    Q3  = DF[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    # Store valid-range min/max before clipping (used in evaluation)
    DF[col] = DF[col].clip(lower=lower, upper=upper)

print('Outlier clipping complete.')

Outlier clipping complete.


## 7. Categorical Encoding

In [7]:
DF['Passes Ro3']       = DF['Passes Ro3'].map({'N': 0, 'Y': 1}).astype(float)
DF['Molecular Species'] = DF['Molecular Species'].map(
    {'NEUTRAL': 0, 'BASE': 1, 'ACID': 2, 'ZWITTERION': 3}
).astype(float)
print('Categorical encoding done.')

Categorical encoding done.


## 8. SMILES Tokenizer

Character-level tokenizer with special tokens: `<pad>`, `<start>`, `<end>`.

In [8]:
smiles_list = DF['Smiles'].dropna().tolist()

# Build vocabulary from all unique characters in dataset
unique_chars = sorted(set(ch for smi in smiles_list for ch in smi))
max_length   = max(len(smi) for smi in smiles_list) + 2  # +2 for <start>/<end>

# Character ↔ index mappings
char_to_idx = {'<pad>': 0, '<start>': 1, '<end>': 2}
for i, ch in enumerate(unique_chars, start=3):
    char_to_idx[ch] = i
idx_to_char = {v: k for k, v in char_to_idx.items()}

VOCAB_SIZE = len(char_to_idx)
print(f'Vocabulary size:  {VOCAB_SIZE}')
print(f'Max SMILES length: {max_length} (with start/end tokens)')
print(f'Unique characters: {unique_chars}')

Vocabulary size:  53
Max SMILES length: 445 (with start/end tokens)
Unique characters: ['#', '%', '(', ')', '+', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '=', '@', 'A', 'B', 'C', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'R', 'S', 'Z', '[', '\\', ']', 'a', 'b', 'c', 'g', 'i', 'l', 'n', 'o', 'p', 'r', 's']


In [9]:
def encode_smiles(smi, char_to_idx, max_len):
    """Encode a SMILES string to a padded integer token sequence."""
    tokens  = [char_to_idx['<start>']]
    tokens += [char_to_idx.get(ch, char_to_idx['<pad>']) for ch in smi]
    tokens += [char_to_idx['<end>']]
    tokens += [char_to_idx['<pad>']] * (max_len - len(tokens))  # right-pad
    return tokens[:max_len]

DF['Encoded_Smiles'] = [encode_smiles(s, char_to_idx, max_length) for s in smiles_list]
print(f'SMILES encoding complete. Example:')
print(f'  Original:  {smiles_list[0]}')
print(f'  Encoded:   {DF["Encoded_Smiles"].iloc[0][:15]}...')

SMILES encoding complete. Example:
  Original:  CC#CCC(C)[C@H](O)/C=C/[C@@H]1[C@H]2C/C(=C/CCCC(=O)O)C[C@H]2C[C@H]1O.NC(CO)(CO)CO
  Encoded:   [1, 25, 25, 3, 25, 25, 25, 5, 25, 6, 39, 25, 22, 28, 41]...


## 9. Molecular Formula Parsing → Atom Feature Matrix

Parse each formula to extract per-element atom counts, ordered canonically via topological sort.

In [10]:
FORMULA_PATTERN = re.compile(r'([A-Z][a-z]?)(\d*)')
formulas = DF['Molecular Formula'].dropna().tolist()

# Determine canonical element ordering via topological sort (Kahn's algorithm)
before_graph = defaultdict(set)
in_degree     = defaultdict(int)
all_elements  = set()

for formula in formulas:
    elements = [el for el, _ in FORMULA_PATTERN.findall(formula)]
    all_elements.update(elements)
    for i in range(len(elements)):
        for j in range(i + 1, len(elements)):
            a, b = elements[i], elements[j]
            if b not in before_graph[a]:
                before_graph[a].add(b)
                in_degree[b] += 1
            in_degree.setdefault(a, 0)

queue = deque([el for el in all_elements if in_degree[el] == 0])
ordered_elements = []
while queue:
    el = queue.popleft()
    ordered_elements.append(el)
    for neighbor in before_graph[el]:
        in_degree[neighbor] -= 1
        if in_degree[neighbor] == 0:
            queue.append(neighbor)

if len(ordered_elements) < len(all_elements):
    print('⚠️  Conflicting element orderings detected — using partial order')
else:
    print(f'✅ Canonical element order ({len(ordered_elements)} elements):')
    print(' → '.join(ordered_elements))

# Build atom count matrix
element_value_lists = {el: [] for el in ordered_elements}
for formula in formulas:
    counts = {el: 0 for el in ordered_elements}
    for el, val in FORMULA_PATTERN.findall(formula):
        if el in counts:
            counts[el] += int(val) if val else 1
    for el in ordered_elements:
        element_value_lists[el].append(counts[el])

for el in ordered_elements:
    DF[el] = element_value_lists[el]

atom_cols_ordered = ordered_elements
atom_features = DF[atom_cols_ordered]
max_atom_vals = atom_features.max().values.astype(np.float32)
min_atom_vals = atom_features.min().values.astype(np.float32)
print(f'Atom feature matrix: {atom_features.shape}')

✅ Canonical element order (25 elements):
C → H → Ga → B → Bi → Al → Ba → Ag → Br → Ca → Cl → Cs → F → I → Li → Mg → K → N → Na → O → P → Rb → S → Sr → Zn
Atom feature matrix: (2407622, 25)


## 10. Normalization & Final Preparation

In [11]:
# Drop raw string columns (no longer needed)
DF.drop(columns=['Smiles', 'Molecular Formula'], inplace=True)

CONDITION_COLS = [
    'Molecular Weight', 'AlogP', 'Polar Surface Area', 'HBA', 'HBD',
    '#RO5 Violations', '#Rotatable Bonds', 'QED Weighted', 'CX LogP',
    'CX LogD', 'Aromatic Rings', 'Heavy Atoms', 'Np Likeness Score',
    'Passes Ro3', 'Molecular Species'
]

ATOM_COLS = atom_cols_ordered  # 25 elements in canonical order

NORMALIZE_COLS = CONDITION_COLS + ATOM_COLS

scaler = MinMaxScaler()
DF[NORMALIZE_COLS] = scaler.fit_transform(DF[NORMALIZE_COLS])

print('Normalization complete.')
print(DF.info())

Normalization complete.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2407622 entries, 0 to 2407621
Data columns (total 41 columns):
 #   Column              Dtype  
---  ------              -----  
 0   Molecular Weight    float64
 1   AlogP               float64
 2   Polar Surface Area  float64
 3   HBA                 float64
 4   HBD                 float64
 5   #RO5 Violations     float64
 6   #Rotatable Bonds    float64
 7   Passes Ro3          float64
 8   QED Weighted        float64
 9   CX LogP             float64
 10  CX LogD             float64
 11  Aromatic Rings      float64
 12  Heavy Atoms         float64
 13  Np Likeness Score   float64
 14  Molecular Species   float64
 15  Encoded_Smiles      object 
 16  C                   float64
 17  H                   float64
 18  Ga                  float64
 19  B                   float64
 20  Bi                  float64
 21  Al                  float64
 22  Ba                  float64
 23  Ag                  float64
 24  

## 11. Custom Layers: ARGRUCell & MiniBatchDiscrimination

In [12]:
GRUCell = tf.keras.layers.GRUCell
RNN     = tf.keras.layers.RNN


class ARGRUCell(tf.keras.layers.Layer):
    """
    Autoregressive GRU Cell for character-level SMILES generation.

    At each time step:
      1. Takes per-step conditioning vector as input
      2. Concatenates it with the previous token's soft embedding
      3. Runs one GRU step
      4. Projects to vocabulary logits
      5. Converts logits → probabilities → soft token embedding (fed to next step)

    This autoregressive feedback loop allows the cell to condition each
    generated character on all previously generated characters.
    """

    def __init__(self, gru_units, vocab_size, token_emb_dim, **kwargs):
        super().__init__(**kwargs)
        self._gru_units      = int(gru_units)
        self._vocab_size     = int(vocab_size)
        self._token_emb_dim  = int(token_emb_dim)
        self.gru_cell        = None
        self.logit_proj      = None
        self.input_proj      = None
        self.token_embeddings = None
        self.start_token     = None

    @property
    def state_size(self):
        """Required by Keras RNN: (hidden_state, prev_token_embedding)"""
        return (self._gru_units, self._token_emb_dim)

    @property
    def output_size(self):
        """Output is vocabulary logits at each step."""
        return self._vocab_size

    def build(self, input_shape):
        self.gru_cell         = GRUCell(self._gru_units)
        self.logit_proj       = Dense(self._vocab_size, name=self.name + '_logit_proj')
        self.input_proj       = Dense(self._token_emb_dim, activation=tf.nn.gelu,
                                      name=self.name + '_input_proj')
        self.token_embeddings = self.add_weight(
            shape=(self._vocab_size, self._token_emb_dim),
            initializer='glorot_uniform', trainable=True,
            name=self.name + '_token_embeddings'
        )
        self.start_token = self.add_weight(
            shape=(1, self._token_emb_dim),
            initializer='zeros', trainable=True,
            name=self.name + '_start_token'
        )
        super().build(input_shape)

    def call(self, inputs, states, training=None):
        """
        Args:
            inputs: (batch, cond_proj_dim) — per-step conditioning
            states: [h_prev, prev_token_emb]
        Returns:
            logits: (batch, vocab_size)
            new_states: [new_h, next_token_emb]
        """
        h_prev         = states[0]
        prev_token_emb = states[1] if len(states) > 1 else tf.tile(self.start_token, [tf.shape(h_prev)[0], 1])

        # Fuse conditioning with previous token embedding
        inp      = tf.concat([prev_token_emb, inputs], axis=-1)
        inp      = self.input_proj(inp)

        # GRU update
        gru_out, [new_h] = self.gru_cell(inp, [h_prev], training=training)

        # Project to vocab; compute soft token embedding for next step
        logits         = self.logit_proj(gru_out)
        probs          = tf.nn.softmax(logits, axis=-1)
        next_token_emb = tf.matmul(probs, self.token_embeddings)

        return logits, [new_h, next_token_emb]

In [13]:
class MiniBatchDiscrimination(tf.keras.layers.Layer):
    """
    MiniBatch Discrimination — prevents mode collapse by allowing the
    discriminator to compare samples within a batch.

    Projects each sample into a kernel space and computes pairwise
    distances, then appends diversity features to the discriminator's
    representation.
    """

    def __init__(self, num_kernels=100, kernel_dim=5, epsilon=1e-6, **kwargs):
        super().__init__(**kwargs)
        self.num_kernels = num_kernels
        self.kernel_dim  = kernel_dim
        self.epsilon     = epsilon

    def build(self, input_shape):
        self.T = self.add_weight(
            shape=(input_shape[-1], self.num_kernels * self.kernel_dim),
            initializer='glorot_uniform', trainable=True, name='T_weight'
        )
        super().build(input_shape)

    def call(self, x):
        batch_size = tf.shape(x)[0]
        M  = tf.reshape(tf.matmul(x, self.T), (batch_size, self.num_kernels, self.kernel_dim))
        M1 = tf.expand_dims(M, axis=3)           # (B, K, D, 1)
        M2 = tf.transpose(M1, [3, 1, 2, 0])      # (1, K, D, B)
        abs_diff          = tf.reduce_sum(tf.abs(M1 - M2), axis=2)  # (B, K, B)
        minibatch_features = tf.reduce_sum(tf.exp(-abs_diff), axis=2) - 1.0  # exclude self
        return tf.concat([x, minibatch_features], axis=1)

## 12. Generator Architecture

In [14]:
def build_generator(noise_dim=90, cond_dim=15, atom_output_dim=25,
                    smiles_seq_len=89, vocab_size=50, smiles_embedding_dim=64):
    """
    Conditional Generator.
    Returns: (model, smiles_branch_layers)
      smiles_branch_layers — list of layer objects that belong to the SMILES
      generation branch, collected by reference during build so we can filter
      their variables precisely in the training loop (no fragile string matching).
    """
    noise_input = Input(shape=(noise_dim,), name='noise_input')
    cond_input  = Input(shape=(cond_dim,),  name='condition_input')

    # Shared trunk
    x = Concatenate()([noise_input, cond_input])
    x = Dense(256, activation=tf.nn.leaky_relu)(x)
    x = Dropout(0.3)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.2)(x)
    x = Dense(256, activation=tf.nn.leaky_relu)(x)
    x = Dense(128, activation='relu')(x)

    # Atom output branch
    atom_output = Dense(atom_output_dim, activation='relu', name='atom_output')(x)

    # MLP for per-step scalar conditioning
    mlp_h = x
    mlp_dense_layers = []
    for _ in range(4):
        d = Dense(256, activation=tf.nn.gelu)
        mlp_h = d(mlp_h)
        mlp_h = Dropout(0.1)(mlp_h)
        mlp_dense_layers.append(d)
    mlp_proj     = Dense(smiles_seq_len)
    mlp_out      = mlp_proj(mlp_h)
    mlp_out_exp  = Lambda(lambda t: tf.expand_dims(t, -1))(mlp_out)

    # Per-step conditioning
    rep_cond      = RepeatVector(smiles_seq_len)(cond_input)
    rep_atom      = RepeatVector(smiles_seq_len)(atom_output)
    per_step_cond = Concatenate()([rep_cond, rep_atom, mlp_out_exp])
    step_proj     = Dense(smiles_embedding_dim, activation=tf.nn.gelu)
    per_step_proj = step_proj(per_step_cond)

    # Initial GRU hidden state
    gru_units = max(128, smiles_embedding_dim)
    init_dense = Dense(gru_units, activation='tanh')
    init_h     = init_dense(x)
    init_tok   = Lambda(lambda t: tf.zeros_like(t))(per_step_proj[:, 0, :])

    # Autoregressive GRU
    ar_cell       = ARGRUCell(gru_units=gru_units, vocab_size=vocab_size,
                              token_emb_dim=smiles_embedding_dim, name='ar_gru')
    rnn_layer     = RNN(ar_cell, return_sequences=True)
    smiles_logits = rnn_layer(per_step_proj, initial_state=[init_h, init_tok])
    smiles_output = Lambda(lambda t: t, name='smiles_output')(smiles_logits)

    model = Model(inputs=[noise_input, cond_input],
                  outputs=[atom_output, smiles_output],
                  name='Generator')

    # Collect SMILES branch layers by object reference — used in train_step
    # to get their variables without relying on fragile name-string matching
    smiles_layers = [ar_cell, rnn_layer, step_proj, mlp_proj] + mlp_dense_layers

    return model, smiles_layers

## 13. Discriminator Architecture

**Bug fixed from original:** The original code had two Conv1D branches both assigned to `x1`, orphaning the first branch. This version has a single, clean 4-layer Conv1D branch.

In [15]:
def build_discriminator(seq_len, vocab_size, atom_dim=25, cond_dim=15):
    """
    Conditional Discriminator:
      Inputs: condition vector + atom composition + SMILES one-hot sequence
      Output: real/fake probability

    Architecture:
      - SMILES branch: 4-layer Conv1D → GlobalAveragePooling
      - Atom branch:   Dense
      - Condition branch: Dense
      - Merged → Dense → MiniBatch Discrimination → output

    Bug fix: original had two Conv1D chains both named x1,
    orphaning the first 2-layer chain. Now uses a single clean pipeline.
    """
    # --- SMILES branch (Conv1D) ---
    smiles_input = layers.Input(shape=(seq_len, vocab_size), name='smiles_input')
    x1 = layers.Conv1D(128, kernel_size=11, padding='same')(smiles_input)
    x1 = layers.LeakyReLU(0.1)(x1)
    x1 = layers.Dropout(0.3)(x1)
    x1 = layers.Conv1D(256, kernel_size=7, padding='same', activation='relu')(x1)
    x1 = layers.Dropout(0.3)(x1)
    x1 = layers.Conv1D(128, kernel_size=5, padding='same')(x1)
    x1 = layers.LeakyReLU(0.1)(x1)
    x1 = layers.Dropout(0.3)(x1)
    x1 = layers.Conv1D(64, kernel_size=3, padding='same')(x1)
    x1 = layers.LeakyReLU(0.1)(x1)
    x1 = layers.GlobalAveragePooling1D()(x1)  # → (batch, 64)

    # --- Atom branch ---
    atom_input = layers.Input(shape=(atom_dim,), name='atom_input')
    x2 = layers.Dense(256)(atom_input)
    x2 = layers.LeakyReLU(0.1)(x2)
    x2 = layers.Dropout(0.3)(x2)

    # --- Condition branch ---
    cond_input = layers.Input(shape=(cond_dim,), name='cond_input')
    x3 = layers.Dense(128)(cond_input)
    x3 = layers.LeakyReLU(0.1)(x3)
    x3 = layers.Dropout(0.3)(x3)

    # --- Merge + MiniBatch Discrimination ---
    x = layers.Concatenate()([x1, x2, x3])
    x = layers.Dense(128)(x)
    x = layers.LeakyReLU(0.1)(x)
    x = layers.Dropout(0.3)(x)
    x = MiniBatchDiscrimination(num_kernels=50, kernel_dim=5)(x)
    x = layers.Dense(64)(x)

    # Output: sigmoid probability, cast to float32 (required for mixed precision)
    out = layers.Dense(1, activation='sigmoid', dtype='float32')(x)

    return tf.keras.Model(
        inputs=[cond_input, atom_input, smiles_input],
        outputs=out,
        name='Discriminator'
    )

## 14. Dataset Preparation

In [16]:
# ─── Config ───────────────────────────────────────────────────────────
NUM_SHARDS    = 50      # total shards for incremental training
SHARD_INDEX   = 0       # change to iterate over shards (0–49)
USE_FULL_DATA = True   # set True to train on full dataset
BATCH_SIZE    = 256
# ──────────────────────────────────────────────────────────────────────

def load_shard(df, use_full=False, shard_index=0, num_shards=50):
    """Return a shard of the DataFrame for incremental training."""
    if use_full:
        print(f'Using full dataset: {len(df):,} rows')
        return df
    total   = len(df)
    size    = total // num_shards
    start   = shard_index * size
    end     = (shard_index + 1) * size if shard_index < num_shards - 1 else total
    print(f'Shard {shard_index + 1}/{num_shards} — rows {start:,} to {end:,} ({end-start:,} samples)')
    return df.iloc[start:end].reset_index(drop=True)

shard_df = load_shard(DF, use_full=USE_FULL_DATA, shard_index=SHARD_INDEX, num_shards=NUM_SHARDS)
train_df, test_df = train_test_split(shard_df, test_size=0.2, random_state=42)

X_train_cond  = tf.convert_to_tensor(train_df[CONDITION_COLS].values.astype('float32'))
X_test_cond   = tf.convert_to_tensor(test_df[CONDITION_COLS].values.astype('float32'))
Y_train_atoms = tf.convert_to_tensor(train_df[ATOM_COLS].values.astype('float32'))
Y_test_atoms  = tf.convert_to_tensor(test_df[ATOM_COLS].values.astype('float32'))
Y_train_smiles = tf.convert_to_tensor(pad_sequences(train_df['Encoded_Smiles'], padding='post'), dtype=tf.int32)
Y_test_smiles  = tf.convert_to_tensor(pad_sequences(test_df['Encoded_Smiles'],  padding='post'), dtype=tf.int32)

train_dataset = (tf.data.Dataset
    .from_tensor_slices((X_train_cond, Y_train_atoms, Y_train_smiles))
    .shuffle(len(X_train_cond))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE))

print(f'Train batches: {len(list(train_dataset))}')
print(f'Condition dim: {X_train_cond.shape[1]}  |  Atom dim: {Y_train_atoms.shape[1]}')
print(f'SMILES seq len: {Y_train_smiles.shape[1]}  |  Vocab size: {VOCAB_SIZE}')

Using full dataset: 2,407,622 rows


I0000 00:00:1781522596.506348      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1781522596.509105      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Train batches: 7524
Condition dim: 15  |  Atom dim: 25
SMILES seq len: 445  |  Vocab size: 53


## 15. Evaluation Utilities

In [17]:
def decode_smiles(token_ids_batch, idx_to_char, eos_token=2):
    """Decode a batch of token ID sequences back to SMILES strings."""
    results = []
    for ids in token_ids_batch:
        chars = []
        for i in ids:
            i = int(i)
            if i == eos_token or i == 0:  # stop at <end> or <pad>
                break
            ch = idx_to_char.get(i, '')
            if ch not in ('<start>', '<end>', '<pad>', ''):
                chars.append(ch)
        results.append(''.join(chars))
    return results


# Atom vocab: atomic number → column index in atom feature matrix
ATOM_VOCAB = {
    6: 0,  7: 1,  8: 2,  9: 3,  15: 4, 16: 5,  17: 6,  35: 7,  53: 8,
    5: 9,  14: 10, 11: 11, 12: 12, 19: 13, 20: 14, 13: 15, 30: 16,
    56: 17, 83: 18, 47: 19, 31: 20, 37: 21, 55: 22, 38: 23, 3: 24
}


def extract_atom_features(smiles, atom_dim=25):
    """Convert a SMILES string to an atom count vector using ATOM_VOCAB."""
    counts = np.zeros(atom_dim, dtype=np.float32)
    mol    = Chem.MolFromSmiles(smiles)
    if mol is not None:
        for atom in mol.GetAtoms():
            Z = atom.GetAtomicNum()
            if Z in ATOM_VOCAB:
                counts[ATOM_VOCAB[Z]] += 1
    return counts


def evaluate_generator(generator, condition_batch, idx_to_char, noise_dim, real_atoms=None):
    """Generate molecules and compute quality metrics."""
    batch_size = condition_batch.shape[0]
    noise      = tf.random.normal((batch_size, noise_dim))
    cond       = tf.convert_to_tensor(condition_batch, dtype=tf.float32)

    fake_atoms, fake_smiles_logits = generator([noise, cond], training=False)
    fake_ids    = tf.argmax(fake_smiles_logits, axis=-1).numpy()
    fake_onehot = tf.one_hot(fake_ids, depth=len(idx_to_char))
    decoded     = decode_smiles(fake_ids, idx_to_char)

    validity    = sum(is_valid_smiles(s) for s in decoded) / len(decoded)
    uniqueness  = len(set(decoded)) / len(decoded)

    # Denormalize atom predictions for comparison
    max_t = tf.constant(max_atom_vals, dtype=fake_atoms.dtype)
    min_t = tf.constant(min_atom_vals, dtype=fake_atoms.dtype)
    fake_atoms_denorm = np.rint((fake_atoms * (max_t - min_t) + min_t).numpy()).astype(int)

    atom_validity   = np.mean((fake_atoms_denorm >= min_atom_vals) & (fake_atoms_denorm <= max_atom_vals)) * 100
    atom_uniqueness = len(set(map(tuple, fake_atoms_denorm))) / len(fake_atoms_denorm) * 100
    atom_exact, atom_mae = 0, None

    if real_atoms is not None:
        real_np = np.array(real_atoms).astype(int)
        atom_exact = np.mean([np.array_equal(fake_atoms_denorm[i], real_np[i]) for i in range(len(real_np))]) * 100
        atom_mae   = mean_absolute_error(real_np, fake_atoms_denorm)

    return dict(validity=validity, uniqueness=uniqueness, generated_smiles=decoded,
                fake_atoms=fake_atoms_denorm, fake_smiles_onehot=fake_onehot,
                fake_smiles_logits=fake_smiles_logits,
                atom_validity=atom_validity, atom_uniqueness=atom_uniqueness,
                atom_exact_match=atom_exact, atom_mae=atom_mae)


def evaluate_discriminator(discriminator, cond, real_atoms, real_smiles,
                           fake_atoms, fake_onehot, idx_to_char):
    """Compute discriminator accuracy on real and fake samples."""
    from sklearn.metrics import accuracy_score
    vocab_size   = len(idx_to_char)
    real_oh      = tf.one_hot(tf.cast(real_smiles, tf.int32), depth=vocab_size)
    real_preds   = discriminator([cond, tf.cast(real_atoms, tf.float32), real_oh],   training=False).numpy()
    fake_preds   = discriminator([cond, tf.cast(fake_atoms, tf.float32), fake_onehot], training=False).numpy()
    real_labels  = np.ones_like(real_preds)
    fake_labels  = np.zeros_like(fake_preds)
    all_preds    = np.vstack([real_preds, fake_preds])
    all_labels   = np.vstack([real_labels, fake_labels])
    acc          = accuracy_score(all_labels, (all_preds > 0.5).astype(int))
    return dict(accuracy=acc,
                real_accuracy=(real_preds > 0.5).mean(),
                fake_accuracy=(fake_preds < 0.5).mean())


def print_evaluation(gen_res, disc_res):
    print('\n📊 Evaluation Results')
    print(f'  SMILES Validity:        {gen_res["validity"]*100:.2f}%')
    print(f'  SMILES Uniqueness:      {gen_res["uniqueness"]*100:.2f}%')
    print(f'  Atom Validity:          {gen_res["atom_validity"]:.2f}%')
    print(f'  Atom Uniqueness:        {gen_res["atom_uniqueness"]:.2f}%')
    print(f'  Atom Exact Match:       {gen_res["atom_exact_match"]:.2f}%')
    if gen_res["atom_mae"] is not None:
        print(f'  Atom MAE:               {gen_res["atom_mae"]:.4f}')
    print(f'  Discriminator Accuracy: {disc_res["accuracy"]*100:.2f}%')
    print(f'  Real Accuracy:          {disc_res["real_accuracy"]*100:.2f}%')
    print(f'  Fake Accuracy:          {disc_res["fake_accuracy"]*100:.2f}%')

In [18]:
# Build matrices for a fully differentiable consistency loss that covers
# BOTH single-character (C, N, O, S, P, F, I, H, B, K) and two-character
# (Cl, Br, Na, Ca, Cs, Ba, Bi, Ag, Ga, Al, Mg, Li, Rb, Sr, Zn) element symbols.
#
# Two-character elements are detected via bigram probabilities:
#   P(Cl at position t) = P(char_t = 'C') * P(char_t+1 = 'l')
# Since 'C' alone would ALSO be counted as Carbon, we subtract that bigram
# probability from Carbon's expected count (avoids double counting).

SINGLE_ELEMENT_CHARS = {
    'C': 'C', 'c': 'C',
    'N': 'N', 'n': 'N',
    'O': 'O', 'o': 'O',
    'S': 'S', 's': 'S',
    'P': 'P', 'p': 'P',
    'F': 'F', 'I': 'I', 'H': 'H', 'B': 'B', 'K': 'K',
}

# (first_char, second_char, element_symbol)
TWO_CHAR_ELEMENTS = [
    ('C','l','Cl'), ('B','r','Br'), ('A','g','Ag'), ('G','a','Ga'),
    ('B','i','Bi'), ('A','l','Al'), ('B','a','Ba'), ('C','a','Ca'),
    ('C','s','Cs'), ('M','g','Mg'), ('R','b','Rb'), ('S','r','Sr'),
    ('Z','n','Zn'), ('N','a','Na'), ('L','i','Li'),
]

ATOM_DIM = len(atom_cols_ordered)

# ── Single-character matrix: (VOCAB_SIZE, ATOM_DIM) ──
SINGLE_MATRIX = np.zeros((VOCAB_SIZE, ATOM_DIM), dtype=np.float32)
for ch, element in SINGLE_ELEMENT_CHARS.items():
    if ch in char_to_idx and element in atom_cols_ordered:
        SINGLE_MATRIX[char_to_idx[ch], atom_cols_ordered.index(element)] = 1.0

# ── Two-character (bigram) info: token-index pairs + add/subtract vectors ──
bigram_idx1, bigram_idx2, adjust_rows = [], [], []
for c1, c2, element in TWO_CHAR_ELEMENTS:
    if c1 not in char_to_idx or c2 not in char_to_idx or element not in atom_cols_ordered:
        continue
    row = np.zeros(ATOM_DIM, dtype=np.float32)
    row[atom_cols_ordered.index(element)] += 1.0          # add to e.g. Cl
    single_el = SINGLE_ELEMENT_CHARS.get(c1)               # e.g. 'C' -> Carbon
    if single_el is not None and single_el in atom_cols_ordered:
        row[atom_cols_ordered.index(single_el)] -= 1.0     # subtract from Carbon
    bigram_idx1.append(char_to_idx[c1])
    bigram_idx2.append(char_to_idx[c2])
    adjust_rows.append(row)

SINGLE_MATRIX_T = tf.constant(SINGLE_MATRIX)                              # (V, A)
BIGRAM_IDX1_T   = tf.constant(bigram_idx1, dtype=tf.int32)                # (K,)
BIGRAM_IDX2_T   = tf.constant(bigram_idx2, dtype=tf.int32)                # (K,)
BIGRAM_ADJUST_T = tf.constant(np.stack(adjust_rows), dtype=tf.float32)    # (K, A)

print(f'Single-char elements mapped: {sum(SINGLE_MATRIX.sum(axis=0) > 0)} / {ATOM_DIM}')
print(f'Two-char elements mapped:    {len(bigram_idx1)} / {len(TWO_CHAR_ELEMENTS)}')
print(f'Atom columns: {atom_cols_ordered}')

Single-char elements mapped: 10 / 25
Two-char elements mapped:    15 / 15
Atom columns: ['C', 'H', 'Ga', 'B', 'Bi', 'Al', 'Ba', 'Ag', 'Br', 'Ca', 'Cl', 'Cs', 'F', 'I', 'Li', 'Mg', 'K', 'N', 'Na', 'O', 'P', 'Rb', 'S', 'Sr', 'Zn']


## 16. Training Setup (Multi-GPU)

In [19]:
NOISE_DIM          = 90
CONDITION_DIM      = X_train_cond.shape[1]
ATOM_DIM           = Y_train_atoms.shape[1]
SMILES_SEQ_LEN     = Y_train_smiles.shape[1]
CONSISTENCY_WEIGHT = 0.05

# Adaptive training flags (updated each epoch based on metrics)
update_discriminator = False
update_atom_branch   = True

# ─── Multi-GPU Strategy ───────────────────────────────────────────────────
strategy = tf.distribute.MirroredStrategy()
print(f'Training on {strategy.num_replicas_in_sync} GPU(s)')

# ALL model and optimizer creation MUST be inside strategy.scope()
with strategy.scope():
    generator, smiles_branch_layers = build_generator(
        noise_dim=NOISE_DIM, cond_dim=CONDITION_DIM,
        atom_output_dim=ATOM_DIM, smiles_seq_len=SMILES_SEQ_LEN, vocab_size=VOCAB_SIZE
    )
    smiles_branch_vars = [v for layer in smiles_branch_layers
                          for v in layer.trainable_variables]
    discriminator = build_discriminator(
        seq_len=SMILES_SEQ_LEN, vocab_size=VOCAB_SIZE,
        atom_dim=ATOM_DIM, cond_dim=CONDITION_DIM
    )
    # Separate learning rates: discriminator slower for training stability
    gen_optimizer  = tf.keras.optimizers.Adam(learning_rate=1e-4)
    disc_optimizer = tf.keras.optimizers.Adam(learning_rate=1e-5)
    bce = tf.keras.losses.BinaryCrossentropy(from_logits=False)
    mse = tf.keras.losses.MeanSquaredError()

generator.summary()
discriminator.summary()

# ─── Distributed dataset ─────────────────────────────────────────────────
# Wrap the dataset with strategy.experimental_distribute_dataset so each GPU
# gets its own shard of every batch automatically.
dist_train_dataset = strategy.experimental_distribute_dataset(train_dataset)
print('Models and distributed dataset ready.')

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
Training on 2 GPU(s)


Model: "Generator"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ noise_input         │ (None, 90)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ condition_input     │ (None, 15)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 105)       │          0 │ noise_input[0][0… │
│ (Concatenate)       │                   │            │ condition_input[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │     27,136 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     32,896 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 256)       │     33,024 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 128)       │     32,896 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 256)       │     33,024 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 256)       │          0 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 256)       │     65,792 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 256)       │          0 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 256)       │     65,792 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 256)       │          0 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 256)       │     65,792 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 256)       │          0 │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ atom_output (Dense) │ (None, 25)        │      3,225 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 445)       │    114,365 │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ repeat_vector       │ (None, 445, 15)   │          0 │ condition_input[… │
│ (RepeatVector)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ repeat_vector_1     │ (None, 445, 25)   │          0 │ atom_output[0][0] │
│ (RepeatVector)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 445, 1)    │          0 │ dense_8[0][0]     │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 496,598 (1.89 MB)

 Trainable params: 496,598 (1.89 MB)

...

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ smiles_input        │ (None, 445, 53)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 445, 128)  │     74,752 │ smiles_input[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu         │ (None, 445, 128)  │          0 │ conv1d[0][0]      │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 445, 128)  │          0 │ leaky_re_lu[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 445, 256)  │    229,632 │ dropout_6[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, 445, 256)  │          0 │ conv1d_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 445, 128)  │    163,968 │ dropout_7[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_1       │ (None, 445, 128)  │          0 │ conv1d_2[0][0]    │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_8 (Dropout) │ (None, 445, 128)  │          0 │ leaky_re_lu_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ atom_input          │ (None, 25)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cond_input          │ (None, 15)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 445, 64)   │     24,640 │ dropout_8[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 256)       │      6,656 │ atom_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 128)       │      2,048 │ cond_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_2       │ (None, 445, 64)   │          0 │ conv1d_3[0][0]    │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_3       │ (None, 256)       │          0 │ dense_11[0][0]    │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_4       │ (None, 128)       │          0 │ dense_12[0][0]    │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ leaky_re_lu_2[0]… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_9 (Dropout) │ (None, 256)       │          0 │ leaky_re_lu_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_10          │ (None, 128)       │          0 │ leaky_re_lu_4[0]… │
│ (Dropout)           │                   │            │                 

 Total params: 602,689 (2.30 MB)

 Trainable params: 602,689 (2.30 MB)

 Non-trainable params: 0 (0.00 B)

Models and distributed dataset ready.


In [20]:
# Do a dummy forward pass first, so ALL nested variables exist
# (e.g. ARGRUCell's inner GRUCell only builds its kernel on first call,
# not when the outer layer's build() runs). Only THEN build the optimizers.
with strategy.scope():
    _dummy_noise = tf.zeros((2, NOISE_DIM))
    _dummy_cond  = tf.zeros((2, CONDITION_DIM))
    _dummy_atoms, _dummy_smiles = generator([_dummy_noise, _dummy_cond], training=False)

    _dummy_smiles_oh = tf.one_hot(tf.zeros((2, SMILES_SEQ_LEN), dtype=tf.int32), depth=VOCAB_SIZE)
    _ = discriminator([_dummy_cond, _dummy_atoms, _dummy_smiles_oh], training=False)

    gen_optimizer.build(generator.trainable_variables)
    disc_optimizer.build(discriminator.trainable_variables)

print(f'Generator variables:     {len(generator.trainable_variables)}')
print(f'Discriminator variables: {len(discriminator.trainable_variables)}')
print('Dummy forward pass done, optimizer slot variables pre-built.')

I0000 00:00:1781522700.228924      58 cuda_dnn.cc:529] Loaded cuDNN version 91002


Generator variables:     33
Discriminator variables: 19
Dummy forward pass done, optimizer slot variables pre-built.


## 17. Train Step

In [21]:
with strategy.scope():

    @tf.function
    def train_step(condition, real_atoms, real_smiles,
                    update_discriminator, update_atom_branch):
        """
        update_discriminator / update_atom_branch: tf.float32 scalars (1.0/0.0).

        Single combined generator loss, single gradient computation,
        single apply_gradients call — avoids overlapping slot-variable
        creation across multiple apply_gradients calls on the same optimizer,
        which is incompatible with MirroredStrategy + tf.function.
        """
        batch_size = tf.shape(condition)[0]
        noise      = tf.random.normal((batch_size, NOISE_DIM))

        # ── Discriminator step ──
        with tf.GradientTape() as disc_tape:
            fake_atoms, fake_smiles_logits = generator([noise, condition], training=True)
            real_oh  = tf.one_hot(real_smiles, depth=VOCAB_SIZE, dtype=fake_smiles_logits.dtype)
            real_out = discriminator([condition, real_atoms, real_oh], training=True)
            fake_out = discriminator([condition, fake_atoms, fake_smiles_logits], training=True)
            d_loss   = (bce(tf.ones_like(real_out) * 0.9, real_out)
                      + bce(tf.zeros_like(fake_out), fake_out))

        d_grads = disc_tape.gradient(d_loss, discriminator.trainable_variables)
        d_grads = [tf.zeros_like(v) if g is None else g * update_discriminator
                   for g, v in zip(d_grads, discriminator.trainable_variables)]
        disc_optimizer.apply_gradients(zip(d_grads, discriminator.trainable_variables))

        # ── Generator step (single combined loss, single update) ──
        with tf.GradientTape() as gen_tape:
            fake_atoms, fake_smiles_logits = generator([noise, condition], training=True)
            fake_atoms_f32 = tf.cast(fake_atoms, tf.float32)
            fake_out       = discriminator([condition, fake_atoms, fake_smiles_logits], training=True)

            g_loss_adv = bce(tf.ones_like(fake_out), fake_out)

            g_loss_smiles = tf.reduce_mean(
                tf.keras.losses.sparse_categorical_crossentropy(
                    real_smiles, fake_smiles_logits, from_logits=True
                )
            )

            # Differentiable consistency loss (single + two-char elements)
            probs = tf.nn.softmax(tf.cast(fake_smiles_logits, tf.float32), axis=-1)
            expected_atoms = tf.einsum('btv,va->ba', probs, SINGLE_MATRIX_T)
            probs1 = tf.gather(probs[:, :-1, :], BIGRAM_IDX1_T, axis=2)
            probs2 = tf.gather(probs[:, 1:, :],  BIGRAM_IDX2_T, axis=2)
            bigram_sums = tf.reduce_sum(probs1 * probs2, axis=1)
            expected_atoms += tf.matmul(bigram_sums, BIGRAM_ADJUST_T)

            fake_atoms_denorm = fake_atoms_f32 * (max_atom_vals - min_atom_vals) + min_atom_vals
            consistency_loss  = mse(fake_atoms_denorm, expected_atoms)

            dtype = g_loss_adv.dtype
            # update_atom_branch masks ONLY the adversarial term — SMILES +
            # consistency terms always contribute (Phase 2 behaviour preserved)
            g_loss_total = (g_loss_adv * tf.cast(update_atom_branch, dtype)
                            + tf.cast(g_loss_smiles, dtype)
                            + tf.cast(consistency_loss, dtype) * tf.cast(CONSISTENCY_WEIGHT, dtype))

        g_grads = gen_tape.gradient(g_loss_total, generator.trainable_variables)
        g_grads = [tf.zeros_like(v) if g is None else g
                   for g, v in zip(g_grads, generator.trainable_variables)]
        gen_optimizer.apply_gradients(zip(g_grads, generator.trainable_variables))

        return (tf.cast(d_loss, tf.float32),
                tf.cast(g_loss_total, tf.float32),
                tf.cast(consistency_loss, tf.float32))

## 18. Training Loop

In [ ]:
def train_cgan(dist_dataset, generator, discriminator, epochs=100,
               checkpoint_dir='checkpoints_cgan'):
    os.makedirs(checkpoint_dir, exist_ok=True)

    update_discriminator = False
    update_atom_branch   = True

    d_losses, g_losses, c_losses = [], [], []

    for epoch in range(epochs):
        print(f'\nEpoch {epoch+1}/{epochs}  [update_disc={update_discriminator}, update_atom={update_atom_branch}]')
        ep_d, ep_g, ep_c = [], [], []

        for step, batch in enumerate(dist_dataset):
            cond_b, atom_b, smiles_b = batch
            disc_mask = tf.constant(1.0 if update_discriminator else 0.0, dtype=tf.float32)
            atom_mask = tf.constant(1.0 if update_atom_branch   else 0.0, dtype=tf.float32)
            d_l, g_l, c_l = strategy.run(
                            train_step,
                            args=(cond_b, atom_b, smiles_b, disc_mask, atom_mask)
                        )
            d_val = strategy.reduce(tf.distribute.ReduceOp.SUM, d_l, axis=None).numpy()
            g_val = strategy.reduce(tf.distribute.ReduceOp.SUM, g_l, axis=None).numpy()
            c_val = strategy.reduce(tf.distribute.ReduceOp.SUM, c_l, axis=None).numpy()
            ep_d.append(d_val); ep_g.append(g_val); ep_c.append(c_val)

            if step % 100 == 0:
                print(f'  Step {step:5d} — D: {d_val:.4f}  G: {g_val:.4f}  C: {c_val:.4f}')

        d_losses.append(np.mean(ep_d))
        g_losses.append(np.mean(ep_g))
        c_losses.append(np.mean(ep_c))

        generator.save_weights(os.path.join(checkpoint_dir, f'gen_epoch{epoch+1:03d}.weights.h5'))
        discriminator.save_weights(os.path.join(checkpoint_dir, f'disc_epoch{epoch+1:03d}.weights.h5'))

        EVAL_SIZE = 128
        gen_res  = evaluate_generator(generator, X_test_cond[:EVAL_SIZE], idx_to_char, NOISE_DIM,
                                      real_atoms=Y_test_atoms[:EVAL_SIZE])
        disc_res = evaluate_discriminator(
            discriminator, X_test_cond[:EVAL_SIZE], Y_test_atoms[:EVAL_SIZE], Y_test_smiles[:EVAL_SIZE],
            gen_res['fake_atoms'], gen_res['fake_smiles_onehot'], idx_to_char
        )
        print_evaluation(gen_res, disc_res)

        update_discriminator = bool(disc_res['real_accuracy'] < 0.6 or disc_res['fake_accuracy'] < 0.6)
        update_atom_branch   = not bool(
            gen_res['atom_validity'] > 95
            and gen_res['atom_uniqueness'] > 95
            and gen_res['atom_mae'] is not None
            and gen_res['atom_mae'] < 0.5
        )

        sample_noise = tf.random.normal((4, NOISE_DIM))
        _, sample_logits = generator([sample_noise, X_train_cond[:4]], training=False)
        sample_decoded   = decode_smiles(tf.argmax(sample_logits, axis=-1).numpy(), idx_to_char)
        print('\nSample Generated SMILES:')
        for i, smi in enumerate(sample_decoded):
            valid = '✅' if is_valid_smiles(smi) else '❌'
            print(f'  [{i+1}] {valid} {smi}')

    plt.figure(figsize=(10, 5))
    plt.plot(d_losses, label='Discriminator Loss')
    plt.plot(g_losses, label='Generator Loss')
    plt.plot(c_losses, label='Consistency Loss')
    plt.xlabel('Epoch'); plt.ylabel('Loss')
    plt.title('cGAN Training Losses')
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
    plt.savefig(os.path.join(checkpoint_dir, 'loss_curves.png'), dpi=150)
    plt.show()
    print(f'\n✅ Training complete. Checkpoints saved to: {checkpoint_dir}')


train_cgan(dist_train_dataset, generator, discriminator, epochs=100)


Epoch 1/100  [update_disc=False, update_atom=True]
INFO:tensorflow:Collective all_reduce tensors: 19 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
INFO:tensorflow:Collective all_reduce tensors: 33 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1


I0000 00:00:1781522729.294217     133 cuda_dnn.cc:529] Loaded cuDNN version 91002


INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
  Step     0 — D: 8.7783  G: 11.9287  C: 70.6806
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/repli